# Model 2: Plant Disease Detection (Google Colab Training Driver)

This thin driver notebook automates Google Colab execution for Model 2 (Plant Disease Detection).
All core preprocessing, architecture comparison, evaluation, Grad-CAM, and export logic reside in the modular Python package `ml.disease_detection`.

## 1. Environment Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import tensorflow as tf

print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {tf.test.gpu_device_name()}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("WARNING: No GPU detected! Check Runtime -> Change runtime type -> Hardware accelerator.")

## 2. Clone Repository & Install Dependencies

In [ ]:
%cd /content
# Set your repository URL
!git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git agri
%cd /content/agri

# Install dependencies (Colab preinstalls TF, CUDA, NumPy; install only missing packages without breaking environment)
!pip install -r ml/disease_detection/requirements-colab.txt

## 3. Dataset Download (Kaggle PlantVillage)

In [ ]:
# Authentication: Try Colab Secrets first, then check for ~/.kaggle/kaggle.json
from google.colab import userdata
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print("Notice: Set KAGGLE_USERNAME/KAGGLE_KEY in Colab Secrets or upload kaggle.json to ~/.kaggle/")

!mkdir -p data/disease_detection
# Download and unzip to local runtime NVMe storage (fast I/O during training)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p data/disease_detection/ --unzip

## 4. Run Training Pipeline (Continuous Checkpointing & Drive Sync)

To resume an interrupted session, simply append `--resume` to the command below.

In [ ]:
DRIVE_BACKUP = '/content/drive/MyDrive/agri_disease_checkpoints'

# Execute full two-phase transfer learning with architecture comparison & calibration
!python -u ml/disease_detection/train.py \
    --data-dir data/disease_detection \
    --output-dir reports/disease_detection \
    --artifacts-dir artifacts/disease_detection \
    --backup-dir {DRIVE_BACKUP} \
    --seed 42

## 5. Sync Artifacts and Reports to Google Drive

In [ ]:
# Persist final models and reports to Drive so they are never lost on runtime shutdown
DRIVE_BACKUP = '/content/drive/MyDrive/agri_disease_checkpoints'
!mkdir -p {DRIVE_BACKUP}
!cp -r artifacts/disease_detection {DRIVE_BACKUP}/
!cp -r reports/disease_detection {DRIVE_BACKUP}/
print(f"Successfully synced production artifacts and evaluation reports to: {DRIVE_BACKUP}")

## 6. Verify Exported Model & Test Inference

In [ ]:
!pytest ml/disease_detection/tests/test_inference.py -v